In [1]:
# used the instructions of https://www.sparkcodehub.com/pyspark/use-cases/recommendation-systems
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.ml.evaluation import RegressionEvaluator

# SparkSession (like the wpo)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AnimeRecommender") \
    .config("spark.ui.port", "4040") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.extraJavaOptions", "-Duser.name=admin") \
    .config("spark.driver.extraJavaOptions", "-Dsun.security.auth.login.config=C:/dev/null") \
    .getOrCreate()

In [2]:
#pre processed data
reviews_data = spark.read.csv("../data/preprocessed/reviews.csv", header=True, inferSchema=True)
anime_data = spark.read.csv("../data/preprocessed/animes.csv", header=True, inferSchema=True)

# The profiles need to be represented in numbers.
# Tried using cast like the intstructions but didn't work due to 
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col("user_id").cast("integer"),
    col("anime_id").cast("integer"),
    col("score").cast("float")
)

final_data.show(7)

+-------+--------+-----+
|user_id|anime_id|score|
+-------+--------+-----+
|     32|   34096|  8.0|
|   1104|   34599| 10.0|
|   1825|   28891|  7.0|
|   3796|    2904|  9.0|
|   9589|    4181| 10.0|
|   9872|    2904| 10.0|
|    554|   16664|  6.0|
+-------+--------+-----+
only showing top 7 rows



In [3]:
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

def clean_sparse_data(input_data, min_user_reviews=3, max_user_reviews=float('inf'), 
                      min_anime_reviews=3, max_anime_reviews=float('inf')):

    print(f"Original Row Count: {input_data.count()}")

    # Calculate activity counts
    user_counts = input_data.groupBy("user_id").agg(F.count("anime_id").alias("user_review_count"))
    anime_counts = input_data.groupBy("anime_id").agg(F.count("user_id").alias("anime_review_count"))

    # Define Filters
    # User filter: must be >= min AND <= max
    valid_users = user_counts.filter(
        (F.col("user_review_count") >= min_user_reviews) & 
        (F.col("user_review_count") <= max_user_reviews)
    ).select("user_id")

    # Anime filter: must be >= min AND <= max
    valid_animes = anime_counts.filter(
        (F.col("anime_review_count") >= min_anime_reviews) & 
        (F.col("anime_review_count") <= max_anime_reviews)
    ).select("anime_id")

    # Apply Joins (Inner join keeps only users/anime that pass the filter)
    cleaned_data = input_data.join(valid_users, on="user_id", how="inner") \
                             .join(valid_animes, on="anime_id", how="inner")

    print(f"Cleaned Row Count: {cleaned_data.count()}")
    
    return cleaned_data

Original Row Count: 130519
Cleaned Row Count: 56611


In [5]:
print("hello")